# Pillar 4: Labor Market Distortion (Proxy-Based)

Model 1:
income_it = beta1*mgnrega_it + rainfall + FE + epsilon

Model 2:
agri_yield_it = beta2*mgnrega_it + rainfall + FE + epsilon

Distortion raw metric = beta1 - beta2
Final score inverts this measure so higher score means lower distortion.

In [1]:
from pathlib import Path
import pandas as pd
from analysis_utils import load_and_preprocess, fit_fe_clustered, minmax_normalize, compute_vif, summarize_model

DATA_PATH = Path('Panel_Data 2014-24.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

df = load_and_preprocess(str(DATA_PATH))

In [2]:
formula_income = 'income ~ mgnrega + rainfall + C(region_id) + C(year)'
formula_yield = 'agri_yield ~ mgnrega + rainfall + C(region_id) + C(year)'

res_income, df_income = fit_fe_clustered(formula_income, df)
res_yield, df_yield = fit_fe_clustered(formula_yield, df)

print('Income model summary')
print(res_income.summary())
print('Yield model summary')
print(res_yield.summary())

b1 = res_income.params.get('mgnrega', 0.0)
b2 = res_yield.params.get('mgnrega', 0.0)
distortion_scalar = b1 - b2
print({'beta_income': b1, 'beta_yield': b2, 'distortion': distortion_scalar})

coef_table = pd.DataFrame([
    {'model': 'income', 'term': 'mgnrega', 'coef': b1, 'p_value': res_income.pvalues.get('mgnrega')},
    {'model': 'yield', 'term': 'mgnrega', 'coef': b2, 'p_value': res_yield.pvalues.get('mgnrega')},
    {'model': 'combined', 'term': 'distortion', 'coef': distortion_scalar, 'p_value': None},
])
coef_table.to_csv(OUT_DIR / 'pillar4_coefficients.csv', index=False)
print(coef_table)

vif_income = compute_vif(df_income, ['mgnrega', 'rainfall'])
vif_yield = compute_vif(df_yield, ['mgnrega', 'rainfall'])
vif_income.to_csv(OUT_DIR / 'pillar4_vif_income.csv', index=False)
vif_yield.to_csv(OUT_DIR / 'pillar4_vif_yield.csv', index=False)
print(vif_income)
print(vif_yield)

common_regions = sorted(set(df_income['region_id']).intersection(set(df_yield['region_id'])))
region_m = df[df['region_id'].isin(common_regions)].groupby('region_id')['mgnrega'].mean()
region_raw = distortion_scalar * region_m
region_distortion = minmax_normalize(region_raw)
pillar4_score = (1 - region_distortion).rename('pillar4_distortion_score').reset_index()
pillar4_score.to_csv(OUT_DIR / 'pillar4_distortion_score.csv', index=False)
pillar4_score.head()

Income model summary
                            OLS Regression Results                            
Dep. Variable:                 income   R-squared:                       0.685
Model:                            OLS   Adj. R-squared:                  0.649
Method:                 Least Squares   F-statistic:                     69.03
Date:                Mon, 13 Apr 2026   Prob (F-statistic):           6.08e-70
Time:                        22:34:12   Log-Likelihood:                -25043.
No. Observations:                2560   AIC:                         5.062e+04
Df Residuals:                    2293   BIC:                         5.218e+04
Df Model:                         266                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

{'beta_income': 82.81370528936478, 'beta_yield': -0.018960947442490078, 'distortion': 82.83266623680727}
      model        term       coef   p_value
0    income     mgnrega  82.813705  0.000024
1     yield     mgnrega  -0.018961  0.528969
2  combined  distortion  82.832666       NaN
   variable       vif
1  rainfall  1.002571
0   mgnrega  1.002571
   variable       vif
1  rainfall  1.002571
0   mgnrega  1.002571


C:\Users\Tanishq op\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 266, but rank is 11
  warnings.warn('covariance of constraints does not have full '
C:\Users\Tanishq op\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 266, but rank is 11
  warnings.warn('covariance of constraints does not have full '


,region_id,pillar4_distortion_score
0,ADILABAD_TELANGANA,0.485031
1,AGRA_UTTAR PRADESH,0.705522
2,AJMER_RAJASTHAN,0.173466
3,ALAPPUZHA_KERALA,0.286122
4,ALIGARH_UTTAR PRADESH,0.760702
